# Análisis de las fuentes y amplitudes de ruido · EKF G/F
## Solubilidad variable · desarrollo exclusivamente con LAB019–021

**Pregunta:** ¿qué estructura de ruido y qué amplitudes están respaldadas por estos ensayos, y qué decisiones siguen siendo supuestos?

No se usan LAB016–018 para ajustar, seleccionar fuentes, inicializar búsquedas ni evaluar alternativas. No se modifican los notebooks previos. La química, la solubilidad y las reglas causales del estimador se mantienen. Las muestras químicas de LAB019–021 son **datos de desarrollo** en este estudio; no son una validación independiente.

El estudio separa: (1) reproducción del EKF existente; (2) recalibración de gas compatible con el operador online; (3) ajuste de ruido; (4) validación interna dejando una fermentación fuera. No se interpretará una mejora del gas como prueba de mejores estados químicos.

**Alcance:** análisis exploratorio, no certificación de parámetros verdaderos. Tres fermentaciones limitan la evidencia. Un óptimo en un límite de búsqueda, la dependencia del inicio o innovaciones correlacionadas impiden presentar las amplitudes como identificadas con precisión.

**Ejecución:** Python con NumPy, pandas, SciPy, matplotlib, nbformat y numba. La ejecución entregada usa el entorno base de Anaconda, que ya dispone de numba. El entorno `fermentation` puede necesitar esa dependencia si se elige para ejecutar este notebook. La compilación acelera las mismas ecuaciones y diferencias finitas; no modifica el filtro ni su paso temporal.

In [ ]:
from pathlib import Path
import sys, json, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nbformat
from IPython.display import display, Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
HERE=ROOT/'notebooks/EKF/kalman_analysis'
sys.path.insert(0,str(HERE))
import analysis_core as core
import analysis_plots as plots
OUT=HERE/'results'
PROTECTED=core.protected_hashes(ROOT)
DATA,CHEM,DATA_AUDIT,RAW_AUDIT=core.prepare(ROOT)
assert tuple(DATA)==('LAB019','LAB020','LAB021')
assert set(CHEM.lab)==set(core.LABS)
display(DATA_AUDIT)
print('All optimization and selection inputs:',list(DATA))
print('No holdout simulations or holdout performance metrics are computed.')

## 1. Procesos, ecuaciones y fuentes candidatas

Definimos $r_c=\mu X$, $r_G=\beta_GX/Y_{E/G}$, $r_F=\beta_FX/Y_{E/F}$, $r_d=k_dX$, $r_m=mX$. Entonces:

\[
\dot X=r_c-r_d,\quad \dot N=-r_c/Y_{X/N},\quad
\dot G=-r_c/Y_{X/G}-r_G-p_Gr_m,\quad
\dot F=-r_c/Y_{X/F}-r_F-p_Fr_m,\quad
\dot E=Y_{E/G}r_G+Y_{E/F}r_F.
\]

$p_G=G/(G+F)$ y $p_F=F/(G+F)$, con las protecciones de denominador del código. La producción es $q_{prod}=r_c/12.72+(Y_{E/G}r_G+Y_{E/F}r_F)/1.05$.

Cada columna de $L$ expresa el efecto de perturbar un flujo. Orden de columnas: crecimiento, fermentación G, fermentación F, CO₂ adicional, mortalidad, mantenimiento. Orden de filas: $[X,N,G,F,E,C_d,I]$. Mortalidad afecta directamente solo a X; mantenimiento consume G/F sin producción directa de etanol ni CO₂ en estas ecuaciones. La cuarta fuente es una discrepancia añadida, no un proceso químico independiente demostrado.

La selección binaria de absorción/emisión para el ruido es una aproximación local heredada; no se identifica aquí como una ley física exacta. Las perturbaciones son aditivas de flujo: pueden actuar aunque un flujo determinista sea pequeño. No equivalen automáticamente a incertidumbres de parámetros cinéticos. Hay que revisar los recortes y la actividad de cada proceso antes de interpretar una fuente como físicamente identificada.

In [ ]:
SOURCE_TABLE=pd.DataFrame([
 ['Growth','rc = mu X','+X; -N/Yxn; -G/Yxg; -F/Yxf; +CO2','g biomass/(L sqrt(h))'],
 ['Glucose fermentation','rG = betaG X / Yeg','-G; +Yeg E; +CO2','g glucose/(L sqrt(h))'],
 ['Fructose fermentation','rF = betaF X / Yef','-F; +Yef E; +CO2','g fructose/(L sqrt(h))'],
 ['Additional CO2','Production discrepancy','Cd or gas integral through transfer','g CO2/(L sqrt(h))'],
 ['Mortality','rd = kd X','-X only directly','g biomass/(L sqrt(h))'],
 ['Maintenance','rm = m X','-pG G; -pF F','g sugar/(L sqrt(h))'],
],columns=['Source','Model term','Mapping','Diffusion-amplitude units'])
display(SOURCE_TABLE)
display(pd.Series(core.THETA,name='Frozen chemical parameters'))

## 2. Referencia reproducida antes de ajustar

Se conservan las medias RAW de 30 minutos, criterios de cobertura/umbral, temperatura del último minuto completado, volumen remanente, nutriciones y condiciones iniciales del EKF existente. Se preservan $P_0$, Joseph, las diferencias finitas y RK4 a un minuto sobre la malla de propagación de cinco minutos. El acumulador I se reinicia tras cada ventana.

Comparamos los RMSE con los resultados guardados en el notebook existente **solo para LAB019–021**. Esta comprobación no selecciona parámetros. La prueba de desarrollo también compara un paso RK4 de medio minuto para evaluar sensibilidad numérica. No se asimilan muestras químicas. Se incluyen las muestras NUT en RMSE y se conserva la exclusión de G+F sustituido por cero. Los iniciales se muestran, pero no entran al RMSE postinicial.

In [ ]:
ARCHIVED={lab:core.run(DATA[lab]) for lab in core.LABS}
base_metrics=pd.concat([core.diagnostics(DATA,CHEM,lab,ARCHIVED[lab])[0] for lab in core.LABS],ignore_index=True)
reference=nbformat.read(ROOT/'notebooks/EKF/05_zenteno_co2_variable_solubility.executed.ipynb',as_version=4)
reference_rows=None
for cell in reference.cells:
    for output in cell.get('outputs',[]):
        for line in output.get('text','').splitlines():
            if line.startswith('METRICS_VARIABLE_JSON '):reference_rows=json.loads(line.split(' ',1)[1])
assert reference_rows is not None
prior=pd.DataFrame(reference_rows)
for row in base_metrics.itertuples():
    if row.variable in ('G','F'):continue
    expected=prior.loc[prior.LAB.eq(row.LAB)&prior.variable.eq(row.variable)&prior.caso.eq('forecast' if row.variable=='CO2' else 'EKF_')]
    assert len(expected)==1 and abs(expected.RMSE.iloc[0]-row.RMSE)<1e-7
assert base_metrics.loc[base_metrics.variable.eq('N'),'n'].eq(12).all()
refinement=[]
for lab in core.LABS:
    fine=core.run(DATA[lab],step=1/120)
    refinement.append(dict(LAB=lab,max_CO2_difference=float(np.nanmax(abs(fine[2][:,0]-ARCHIVED[lab][2][:,0]))),
        max_physical_state_difference=float(np.max(abs(fine[0][:,:6]-ARCHIVED[lab][0][:,:6])))))
display(base_metrics)
display(pd.DataFrame(refinement))
print('Archived RMSE reproduced; all 12 post-initial YAN observations included per LAB.')

## 3. Qué es fijo y qué se estima

En unidades físicas $Q=L\operatorname{diag}(\sigma_j^2)L^T$. En el código se escala y se añade $Q\Delta t$ a la covarianza propagada. Las amplitudes son desviaciones de difusión por raíz de hora; no son desviaciones estándar de las concentraciones iniciales ni porcentajes.

$P_0$ se mantiene: $\sigma_X=\max(0.03,0.2X_0)$, $\sigma_N=0.015$ g/L, $\sigma_E=2$ g/L, varG=12.5, varF=37.5, cov(G,F)=-12.5; varCd=varI=0. No se añade incertidumbre independiente a temperatura, presión ni dosis. Son limitaciones explícitas.

Los parámetros `growth`, `fermentation` y `gas` son multiplicadores de amplitudes originales. `mortality`, `maintenance` y `R` son amplitudes absolutas. `R` significa aquí la desviación estándar; la covarianza de observación usada es su cuadrado. La alternativa Separate G/F separa los multiplicadores de ambas fermentaciones. No se ajustan libremente 49 entradas de Q.

In [ ]:
display(pd.DataFrame([dict(structure=k,parameters=', '.join(v),count=len(v)) for k,v in core.STRUCTURES.items()]))
display(pd.DataFrame([dict(parameter=k,lower=10**lo,upper=10**hi) for k,(lo,hi) in core.LOG_BOUNDS.items()]))
print('Bounds are exploratory search constraints, not experimentally determined uncertainty intervals.')
plots.raw_noise(RAW_AUDIT)
display(RAW_AUDIT.groupby('LAB')[['raw_SD','naive_SE','lag1','half_difference']].median())

La dispersión dentro de una ventana mezcla ruido instrumental, fluctuaciones reales, tendencia y picos. `naive_SE` supone muestras independientes y **no se utiliza para fijar R**. La autocorrelación y la diferencia entre mitades muestran por qué dividir por la raíz del número de muestras puede ser engañoso. Sin réplicas instrumentales/referencia independiente, el R ajustado será una incertidumbre efectiva, no una certificación metrológica. No se elimina ningún pico ni se introduce un nuevo tratamiento RAW en este estudio.

## 4. Ajuste y validación interna

Se reajusta kLa/ganancia por perfil y ganancia analítica en cada partición, usando únicamente sus laboratorios de entrenamiento. **A diferencia de la calibración archivada con medianas horarias y temperatura offline, estas recalibraciones usan el operador causal de 30 minutos y la inicialización online de etanol.** Se muestra por separado el cambio de gas y el de ruido; no se atribuyen ambos cambios a Q.

Después se optimizan las amplitudes positivas en escala logarítmica con dos inicializaciones deterministas y búsqueda acotada. Un ajuste sin convergencia, en un límite o dependiente del inicio se marca y no constituye un valor identificado definitivo.

El objetivo usa innovaciones previas a corregir: $\frac12\operatorname{mean}[\log(2\pi S_k)+\nu_k^2/S_k]$, con igual peso por LAB. **Es un criterio gaussiano aproximado condicionado a las ventanas elegibles, no la verosimilitud exacta de todos los datos:** hay recortes, no linealidades y selección de ventanas dependiente del CO₂ medido. También se reporta RMSE sobre todas las ventanas válidas.

Se dejan fuera LAB019, LAB020 y LAB021 por turnos. Dentro de cada ajuste de ruido también se deja fuera ese laboratorio de la calibración de gas. Las métricas químicas pueden informar la decisión de desarrollo, pero no entran al objetivo numérico ni a las correcciones. El menor criterio de CO₂ identifica un **candidato**, no una aprobación automática de su estructura.

In [ ]:
fingerprint=core.study_fingerprint(DATA,CHEM)
cache=OUT/'study.json'
if cache.exists() and json.loads(cache.read_text()).get('fingerprint')==fingerprint:
    STUDY=json.loads(cache.read_text());pack=np.load(OUT/'development_trajectories.npz');RECORDS={}
    for structure in [*core.STRUCTURES,'Original noise']:
        for lab in core.LABS:
            key=structure.replace(' ','_').replace('+','plus').replace('/','_')+'__'+lab
            RECORDS[structure,lab]=tuple(pack[key+'__'+suffix] for suffix in ('frame','covariance','diagnostics'))
    print('Loaded verified results for these exact training data and analysis code.')
else:
    STUDY,RECORDS=core.perform_study(DATA,CHEM,OUT,workers=3)
assert tuple(STUDY['allowed_labs'])==core.LABS
SCORES=pd.DataFrame(STUDY['scores']);METRICS=pd.DataFrame(STUDY['metrics']);CHECKS=pd.DataFrame(STUDY['checks'])
LEADER=STUDY['leader']
display(pd.DataFrame(STUDY['gas']))
display(SCORES)
print('Provisional CO2-score candidate:',LEADER)
print('All labels are development or internal validation, never external holdout.')
# Separate the gas-refit effect from the noise-fit effect on development errors.
comparison=[base_metrics.assign(case='Archived gas + original noise')]
for name in ['Original noise','Current']:
    comparison.append(METRICS.loc[METRICS.fold.eq('All development')&METRICS.structure.eq(name)].assign(case='Refitted gas + '+name))
display(pd.concat(comparison).groupby(['variable','unit','case']).RMSE.mean().unstack('case').round(4))


## 5. Relación entre procesos y fuentes

La actividad se calcula sobre la trayectoria de referencia con el gas reajustado y el ruido original. Los valores integrados son descriptivos de esa estimación, no mediciones independientes de los flujos. Una fuente aditiva puede seguir perturbando el sistema aunque el flujo determinista se anule; esto debe distinguirse de atribuirle una incertidumbre a un parámetro cinético.

In [ ]:
display(plots.mapping(DATA,RECORDS,np.array([STUDY['gas'][0]['kLa'],STUDY['gas'][0]['gain']])))
ACTIVITY=plots.activity(DATA,RECORDS)
display(ACTIVITY)
ACTIVITY.to_csv(OUT/'process_activity.csv',index=False)

## 6. Rendimiento fuera del ajuste de cada partición

La selección de una estructura no puede basarse únicamente en la mejora del CO₂. Los RMSE de estados químicos se presentan por variable y unidad, sin sumarlos arbitrariamente. Estos tres conjuntos de validación interna son pequeños; la elección final necesita ensayos independientes.

In [ ]:
plots.score_figures(STUDY)
CV_METRICS=METRICS.loc[METRICS.fold.ne('All development')]
display(CV_METRICS.groupby(['structure','variable','unit']).RMSE.mean().unstack('structure').round(4))
display(CV_METRICS.pivot(index=['LAB','variable','unit'],columns='structure',values='RMSE').round(4))

## 7. Coherencia de las incertidumbres y restricciones

La cobertura química mostrada compara las muestras con ±1.96 desviaciones del **estado**; no incluye una incertidumbre analítica independiente y no es una cobertura calibrada de observación. No se optimiza para alcanzar 95 %. El NIS y las bandas de innovaciones son diagnósticos aproximados: los recortes y la selección de datos impiden aplicar sin reservas una interpretación chi-cuadrado ideal. La autocorrelación se calcula conservando los huecos entre ventanas no asimiladas.

In [ ]:
plots.innovation_figures(STUDY)
display(CHECKS.drop(columns='acf').round(4))
display(METRICS.groupby(['fold','structure','variable']).coverage95.mean().unstack('variable').round(3))
assert CHECKS.min_eigenvalue.min()>-1e-7

## 8. Estabilidad, límites e identificabilidad

Se comparan las amplitudes obtenidas con diferentes subconjuntos y diferentes inicializaciones. Los barridos son **condicionales**, manteniendo los otros parámetros fijos: no son perfiles reoptimizados ni intervalos de confianza. La SVD y el alineamiento de sensibilidades de predicción/varianza son comprobaciones locales de confundibilidad, no una demostración de identificabilidad global. La firma combina predicciones normalizadas por 0.04 y log-varianzas, por lo que su escala es explícitamente diagnóstica.

In [ ]:
PARAMETERS=plots.parameter_figures(STUDY)
fit_rows=[]
for fit in STUDY['fits']:
    fit_rows.append(dict(fold=fit['fold'],structure=fit['structure'],Rsd=fit['Rsd'],bounds=', '.join(fit['at_bound']),
        converged_starts=sum(r['success'] for r in fit['runs']),
        score_spread=max(r['score'] for r in fit['runs'])-min(r['score'] for r in fit['runs']),
        sigma_growth=fit['sigma'][0],sigma_G=fit['sigma'][1],sigma_F=fit['sigma'][2],sigma_CO2=fit['sigma'][3],
        sigma_mortality=fit['sigma'][4],sigma_maintenance=fit['sigma'][5]))
FIT_TABLE=pd.DataFrame(fit_rows);display(FIT_TABLE)
display(PARAMETERS.pivot(index=['structure','parameter'],columns='fold',values='value'))
print('Local sensitivity singular values:',STUDY['sensitivity']['singular_values'])
FIT_TABLE.to_csv(OUT/'fitted_amplitudes.csv',index=False)
# Probe beyond active bounds without using the probes to select a structure.
BOUND_PROBES=[]
for fit in STUDY['fits']:
    if fit['fold']!='All development':continue
    names=fit['names'];xx=np.array(fit['x']);bounds=core.bounds_for(names)
    for name in fit['at_bound']:
        j=names.index(name);direction=-1 if xx[j]-bounds[j][0]<bounds[j][1]-xx[j] else 1
        for factor in [2.,4.]:
            trial=xx.copy();trial[j]+=direction*np.log10(factor)
            score=core.objective(trial,names,DATA,core.LABS,np.array([STUDY['gas'][0]['kLa'],STUDY['gas'][0]['gain']]))
            BOUND_PROBES.append(dict(structure=fit['structure'],parameter=name,factor_beyond_bound=factor,
                score=score,delta_from_fit=score-fit['score']))
BOUND_PROBES=pd.DataFrame(BOUND_PROBES)
display(BOUND_PROBES)
BOUND_PROBES.to_csv(OUT/'bound_sensitivity.csv',index=False)
print('Negative score changes beyond a bound are evidence that the original search range did not bracket a local minimum.')


## 9. Trayectorias con incertidumbre: efecto del ajuste de ruido

Se comparan el ruido original y el candidato de menor criterio interno de CO₂, ambos con la **misma** calibración de gas sobre todo el desarrollo. Las figuras no son resultados de validación independiente. El CO₂ es previo a la corrección; los estados químicos son posteriores a la actualización cuando corresponde. Se conservan las muestras iniciales y NUT; no se ocultan intervalos negativos de la aproximación gaussiana.

**Lectura de las leyendas:** Original noise significa las matrices originales aplicadas a la calibración de gas reajustada en este análisis. La reproducción con la calibración archivada aparece en la sección 2 y en la tabla que separa gas y ruido. Un cambio de la capa de gas puede inducir trayectorias químicas distintas aunque las matrices de ruido no cambien.


In [ ]:
plots.trajectories(DATA,CHEM,RECORDS,LEADER,'CO2','CO₂')

In [ ]:
plots.trajectories(DATA,CHEM,RECORDS,LEADER,'X','Biomass')

In [ ]:
plots.trajectories(DATA,CHEM,RECORDS,LEADER,'N','YAN')

In [ ]:
plots.trajectories(DATA,CHEM,RECORDS,LEADER,'GF','Glucose + fructose')

In [ ]:
plots.trajectories(DATA,CHEM,RECORDS,LEADER,'E','Ethanol')

## 9b. Comprobaciones que impiden sobreinterpretar el ajuste

Una columna de mortalidad igual a −1 solo en X, con ruido gaussiano simétrico y aditivo, es matemáticamente equivalente en covarianza a una perturbación independiente de biomasa con signo +1. Una mejora con esa columna **no identifica la mortalidad como causa física**; tampoco estima la incertidumbre de kd. El ruido puede corregir biomasa en ambos sentidos.

Los límites influyen en la comparación: si aumentar la amplitud de crecimiento más allá del rango sigue mejorando el criterio, la clasificación entre estructuras queda condicionada por esos límites. Los barridos fuera del rango detectan esta situación, pero no reemplazan una nueva validación con rangos ampliados. No se acepta una estructura definitiva en este notebook.

También revisamos aumentos tardíos de YAN sin nueva nutrición. Son cambios del estimador, no producción biológica de nitrógeno. Las bandas gaussianas pueden cruzar cero y las covarianzas tras recortar los estados son aproximadas.


In [ ]:
anomalies=[]
for label,records in [('Archived gas + original noise',ARCHIVED),('Refitted gas + original noise',{lab:RECORDS['Original noise',lab] for lab in core.LABS}),('Refitted gas + fitted candidate',{lab:RECORDS[LEADER,lab] for lab in core.LABS})]:
    for lab in core.LABS:
        frame=records[lab][0];grid=DATA[lab]['grid'];late=grid>100
        rises=(np.diff(frame[:,1])>1e-8)&(grid[1:]>100)&(DATA[lab]['doses'][1:]==0)
        anomalies.append(dict(case=label,LAB=lab,max_YAN_after_100h_mg_L=float(frame[late,1].max()*1000),positive_YAN_increments_after_100h=int(rises.sum())))
display(pd.DataFrame(anomalies))
pd.DataFrame(anomalies).to_csv(OUT/'late_YAN_diagnostics.csv',index=False)
multistart=[]
for fit in STUDY['fits']:
    if fit['fold']!='All development':continue
    for search in fit['runs']:
        sig,rsd=core.decode(np.array(search['x']),fit['names'])
        multistart.append(dict(structure=fit['structure'],start=search['start'],score=search['score'],converged=search['success'],
            sigma_growth=sig[0],sigma_G=sig[1],sigma_F=sig[2],sigma_CO2=sig[3],sigma_mortality=sig[4],sigma_maintenance=sig[5],Rsd=rsd))
display(pd.DataFrame(multistart))
pd.DataFrame(multistart).to_csv(OUT/'multistart_amplitudes.csv',index=False)


## 10. Conclusiones y decisiones pendientes

La tabla siguiente explicita el criterio de CO₂, los conflictos con estados químicos y las advertencias de cada alternativa. No se publica una nueva calibración oficial ni se modifica la configuración de ningún estimador. La estructura actual tiene fuentes de flujo escogidas, no una validación experimental de cuatro mecanismos independientes.

In [ ]:
cv_score=SCORES.loc[SCORES.fold.ne('All development')].groupby('structure').score.mean()
cv_rmse=CV_METRICS.groupby(['structure','variable']).RMSE.mean().unstack()
DECISION=cv_rmse.join(cv_score.rename('CO2_quasi_score'))
for structure in DECISION.index:
    subset=SCORES.loc[SCORES.structure.eq(structure)]
    DECISION.loc[structure,'any_bound']=bool(subset.at_bound.str.len().gt(0).any())
    DECISION.loc[structure,'all_searches_converged']=bool(subset.converged.all())
display(DECISION.round(4))
reference=DECISION.loc['Current'];candidate=DECISION.loc[LEADER]
text=f"**Candidato por predicción de CO₂:** {LEADER}. Es una clasificación de desarrollo, no una elección definitiva. "
conflicts=[s for s in ['X','N','GF','E'] if candidate[s]>reference[s]]
text+='Estados con mayor RMSE interno que la estructura actual ajustada: '+(', '.join(conflicts) if conflicts else 'ninguno de X, N, GF y E')+'. '
if candidate['any_bound']:text+='Hay parámetros en límites de búsqueda: las amplitudes no pueden declararse identificadas con precisión. '
if not candidate['all_searches_converged']:text+='Alguna búsqueda no convergió; su comparación es provisional. '
text+='Los barridos fuera de límites mejoran algunas alternativas; esta clasificación depende del rango y no decide una estructura definitiva. '
text+='La estabilidad entre inicios, las innovaciones y las coberturas deben evaluarse conjuntamente. Los datos no garantizan una descomposición única entre Q y R.'
display(Markdown(text))
DECISION.to_csv(OUT/'decision_table.csv')
core.verify_protected(ROOT,PROTECTED)
print('Protected existing notebooks, source and configurations unchanged:',len(PROTECTED))
print('Every fitting/selection dataset:',STUDY['allowed_labs'])
print('No LAB016–018 result entered fitting, source selection or diagnostics.')

## Referencias y alcance del respaldo

- [Särkkä y Svensson, Bayesian Filtering and Smoothing, capítulo 16](https://users.aalto.fi/~ssarkka/pub/bfs_book_2023_online.pdf): estimación de parámetros mediante filtros y verosimilitud aproximada.
- [MathWorks, kalman, Algorithms](https://www.mathworks.com/help/control/ref/ss.kalman.html): covarianza de fuentes y transformación mediante su matriz de entrada al sistema.
- [Liger-Belair et al., 2023](https://doi.org/10.1021/acsomega.3c01812): correlación de solubilidad usada por el modelo. Su extrapolación a mosto fermentando y la presión parcial asumida siguen siendo limitaciones.

Estas referencias respaldan métodos y relaciones generales. **No respaldan las amplitudes numéricas elegidas previamente ni prueban que nuestras seis fuentes candidatas sean suficientes.** La transferencia, el mapeo local del ruido y los recortes siguen siendo aproximaciones de esta implementación.